In [3]:
# ===== CELL 1: Setup (run this first, every time you open the notebook) =====
import os, sys, re, io, json, glob, math, time, shutil, zipfile, subprocess, tarfile, importlib.util
import numpy as np, pandas as pd, soundfile as sf, requests, torch
from scipy.signal import resample_poly
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# Run Jupyter from the project directory so downloaded data and results stay local.
PROJECT_DIR = os.path.abspath(os.getcwd())
LOCAL_DATA_DIR = os.path.join(PROJECT_DIR, 'local_data')
OUT_DIR = os.path.join(PROJECT_DIR, 'outputs', 'shriya_aasist')
DATA_DIR = os.path.join(LOCAL_DATA_DIR, 'asvspoof5_data')  # protocol files
WORK_DIR = os.path.join(LOCAL_DATA_DIR, 'work')           # temporary downloads and extracted audio
REPO_DIR = os.path.join(LOCAL_DATA_DIR, 'asvspoof5')      # official code and AASIST weights
for path in [DATA_DIR, WORK_DIR, OUT_DIR]:
    os.makedirs(path, exist_ok=True)
for sub in ['scores/parts', 'keys', 'metrics', 'splits', 'dfadd']:
    os.makedirs(os.path.join(OUT_DIR, sub), exist_ok=True)

ZENODO_URL = 'https://zenodo.org/records/14498691/files'                 # ASVspoof 5
DFADD_URL = 'https://huggingface.co/datasets/isjwdu/DFADD/resolve/main'  # DFADD (updated Apr 2025 zips)

DEV_TARS = [f'flac_D_{s}.tar' for s in ['aa', 'ab', 'ac']]
EVAL_TARS = [f'flac_E_{s}.tar' for s in ['aa', 'ab', 'ac', 'ad', 'ae', 'af', 'ag', 'ah', 'ai', 'aj']]
TRAIN_TARS = [f'flac_T_{s}.tar' for s in ['aa', 'ab', 'ac', 'ad', 'ae']]

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Scoring will run on CPU and may be slow.')
print('Project directory:', PROJECT_DIR)
print('Free local disk: %.0f GB' % (shutil.disk_usage(WORK_DIR).free / 1e9))

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/asvspoof-challenge/asvspoof5.git', REPO_DIR], check=True)
print('Official ASVspoof 5 repo ready.')

GPU: NVIDIA GeForce RTX 4070 SUPER
Project directory: c:\Users\ghazz\Downloads\DeepLearningProj
Free local disk: 315 GB
Official ASVspoof 5 repo ready.


In [4]:
# ===== CELL 2: Helper functions + load the official AASIST model =====
def download(url, dest, retries=5):
    """Download with a progress line. If it gets cut off, rerunning continues where it stopped."""
    for attempt in range(1, retries + 1):
        have = os.path.getsize(dest) if os.path.exists(dest) else 0
        headers = {'Range': f'bytes={have}-'} if have else {}
        try:
            with requests.get(url, headers=headers, stream=True, timeout=60) as r:
                if r.status_code == 416:          # file was already fully downloaded
                    return dest
                r.raise_for_status()
                if r.status_code == 200:          # server sent the whole file, start over
                    have = 0
                total = have + int(r.headers.get('Content-Length', 0))
                done, last = have, 0.0
                with open(dest, 'ab' if have else 'wb') as f:
                    for chunk in r.iter_content(chunk_size=8 << 20):
                        f.write(chunk)
                        done += len(chunk)
                        if total and time.time() - last > 5:
                            print(f'\r  {os.path.basename(dest)}: {100 * done / total:5.1f}% ({done >> 20} MB)', end='')
                            last = time.time()
            if total and os.path.getsize(dest) != total:
                raise IOError('download incomplete')
            print(f'\r  {os.path.basename(dest)}: downloaded ({os.path.getsize(dest) >> 20} MB)          ')
            return dest
        except Exception as e:
            print(f'\n  download problem ({e}); retry {attempt}/{retries}')
            time.sleep(10)
    raise RuntimeError(f'Could not download {url}')


def extract_tar(tar_path, dest):
    os.makedirs(dest, exist_ok=True)
    with tarfile.open(tar_path, 'r:*') as archive:
        archive.extractall(dest)


def find_audio(root, exts=('.flac', '.wav')):
    """Map file ID (name without extension) -> full path, wherever the tar put it."""
    found = {}
    for dirpath, _, files in os.walk(root):
        for fn in files:
            if fn.lower().endswith(exts):
                found[os.path.splitext(fn)[0]] = os.path.join(dirpath, fn)
    return found


def save_tsv(df, path):
    """Write to a temp file first so a crash never leaves a half-written file that looks finished."""
    tmp = path + '.tmp'
    df.to_csv(tmp, sep='\t', index=False)
    os.replace(tmp, path)


CUT = 64600  # official AASIST input: 64,600 samples (~4 s at 16 kHz)


class ScoringDataset(Dataset):
    """Based on my ASVspoof5Dataset, changed to load audio exactly like the official AASIST TestDataset:
    first 64,600 samples, short clips are REPEATED (not zero-padded), and it returns the file ID."""

    def __init__(self, ids, paths):
        self.ids, self.paths = list(ids), list(paths)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        x, sr = sf.read(self.paths[i], dtype='float32')
        if x.ndim > 1:
            x = x.mean(axis=1)                        # stereo -> mono
        assert sr == 16000, f'{self.paths[i]} is {sr} Hz, expected 16000'
        if len(x) == 0:
            x = np.zeros(CUT, dtype='float32')
        elif len(x) >= CUT:
            x = x[:CUT]
        else:
            x = np.tile(x, CUT // len(x) + 1)[:CUT]
        return torch.from_numpy(np.ascontiguousarray(x)), self.ids[i]


def load_aasist():
    base = os.path.join(REPO_DIR, 'Baseline-AASIST')
    spec = importlib.util.spec_from_file_location('aasist_model', os.path.join(base, 'models', 'AASIST.py'))
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    config = json.load(open(os.path.join(base, 'config', 'AASIST_ASVspoof5.conf')))['model_config']
    net = module.Model(config).to(device)
    weights = os.path.join(base, 'models', 'weights', 'AASIST', 'best.pth')
    net.load_state_dict(torch.load(weights, map_location=device))
    return net.eval()


model = load_aasist()
print('Loaded official ASVspoof 5 AASIST checkpoint:', sum(p.numel() for p in model.parameters()), 'parameters')


@torch.no_grad()
def score(ids, paths, batch_size=64):
    loader = DataLoader(ScoringDataset(ids, paths), batch_size=batch_size, shuffle=False,
                        num_workers=min(4, os.cpu_count() or 1), pin_memory=(device == 'cuda'))
    names, logits = [], []
    for x, keys in tqdm(loader, leave=False):
        _, out = model(x.to(device))
        names += list(keys)
        logits.append(out.float().cpu().numpy())
    L = np.concatenate(logits) if logits else np.zeros((0, 2))
    # Official convention: score = bona fide logit (column 1). Higher = more likely real.
    return pd.DataFrame({'filename': names, 'cm-score': L[:, 1],
                         'logit_spoof': L[:, 0], 'logit_bonafide': L[:, 1]})


def part_path(prefix, tar_name):
    return os.path.join(OUT_DIR, 'scores', 'parts', f'{prefix}_{tar_name[:-4]}.tsv')


def run_tar_job(prefix, tar_name, wanted_ids):
    """Download one tar, score its requested audio files, and delete temporary audio."""
    out = part_path(prefix, tar_name)
    if os.path.exists(out):
        print(f'[skip] {tar_name} already scored for {prefix}')
        return
    if shutil.disk_usage(WORK_DIR).free < 20e9:
        print('WARNING: under 20 GB free on the local disk; this tar may not fit.')
    tar_path = os.path.join(WORK_DIR, tar_name)
    ex_dir = tar_path + '_extracted'
    for attempt in (1, 2):
        download(f'{ZENODO_URL}/{tar_name}', tar_path)
        try:
            extract_tar(tar_path, ex_dir)
            break
        except (tarfile.TarError, EOFError):
            print('  tar file looks damaged, downloading it again')
            if os.path.exists(tar_path):
                os.remove(tar_path)
            shutil.rmtree(ex_dir, ignore_errors=True)
    else:
        raise RuntimeError(f'{tar_name} failed twice')
    os.remove(tar_path)
    audio = find_audio(ex_dir)
    todo = sorted(k for k in audio if k in wanted_ids)
    print(f'  {tar_name}: {len(audio)} audio files, scoring {len(todo)}')
    save_tsv(score(todo, [audio[k] for k in todo]), out)
    shutil.rmtree(ex_dir)
    print(f'  saved {os.path.basename(out)} locally')

def merge_parts(prefix, tars, expected_ids, out_name):
    parts = [part_path(prefix, t) for t in tars]
    not_done = [os.path.basename(p) for p in parts if not os.path.exists(p)]
    if not_done:
        raise RuntimeError(f'Not finished yet: {not_done}. Rerun the cell above.')
    df = pd.concat([pd.read_csv(p, sep='\t', dtype={'filename': str}) for p in parts])
    df = df.drop_duplicates('filename')
    df = df[df.filename.isin(expected_ids)].reset_index(drop=True)
    missing = sorted(set(expected_ids) - set(df.filename))
    print(f'{prefix}: scored {len(df)} of {len(expected_ids)} expected files; missing {len(missing)}')
    if missing:
        print('  first missing IDs:', missing[:10])
    save_tsv(df, os.path.join(OUT_DIR, 'scores', out_name))
    return df


# Official ASVspoof 5 metrics (minDCF is the primary metric)
sys.path.insert(0, os.path.join(REPO_DIR, 'evaluation-package'))
from calculate_metrics import calculate_minDCF_EER_CLLR_actDCF

METRICS_FILE = os.path.join(OUT_DIR, 'metrics', 'aasist_metrics.json')


def metrics(scores_df, keys_df, name):
    """keys_df needs columns 'filename' and 'KEY' (bonafide/spoof)."""
    m = scores_df.merge(keys_df[['filename', 'KEY']], on='filename')
    if m.KEY.nunique() < 2:
        print(f'{name}: needs both bona fide and spoof files, skipping')
        return None
    mindcf, eer, cllr, actdcf = calculate_minDCF_EER_CLLR_actDCF(
        m['cm-score'].to_numpy(), m['KEY'].to_numpy(), output_file=os.devnull, printout=False)
    res = {'set': name, 'n_files': len(m), 'n_bonafide': int((m.KEY == 'bonafide').sum()),
           'minDCF': round(float(mindcf), 4), 'EER_%': round(100 * float(eer), 2),
           'CLLR': round(float(cllr), 4), 'actDCF': round(float(actdcf), 4)}
    allm = json.load(open(METRICS_FILE)) if os.path.exists(METRICS_FILE) else {}
    allm[name] = res
    json.dump(allm, open(METRICS_FILE, 'w'), indent=2)
    print(res)
    return res

Loaded official ASVspoof 5 AASIST checkpoint: 297866 parameters


In [5]:
# ===== CELL 3: Protocol files (labels for every file) + integrity check =====
COLS = ['SPEAKER_ID', 'FLAC_FILE_NAME', 'SPEAKER_GENDER', 'CODEC', 'CODEC_Q',
        'CODEC_SEED', 'ATTACK_TAG', 'ATTACK_LABEL', 'KEY', 'TMP']
PROTO_FILES = {'train': 'ASVspoof5.train.tsv', 'dev': 'ASVspoof5.dev.track_1.tsv',
               'eval': 'ASVspoof5.eval.track_1.tsv'}
EXPECTED_ROWS = {'train': 182357, 'dev': 140950, 'eval': 680774}


def find_proto(name):
    hits = glob.glob(os.path.join(DATA_DIR, '**', name), recursive=True)
    return hits[0] if hits else None


if not all(find_proto(f) for f in PROTO_FILES.values()):
    tgz = download(f'{ZENODO_URL}/ASVspoof5_protocols.tar.gz', os.path.join(WORK_DIR, 'ASVspoof5_protocols.tar.gz'))
    extract_tar(tgz, DATA_DIR)
    os.remove(tgz)

print('--- ASVspoof 5 PROTOCOL CHECK ---')
proto = {}
for split, fn in PROTO_FILES.items():
    path = find_proto(fn)
    print(f'[{"EXISTS" if path else "MISSING"}] {fn}')
    assert path, f'{fn} is missing'
    df = pd.read_csv(path, sep=r'\s+', header=None, names=COLS, dtype=str)
    df['filename'] = df['FLAC_FILE_NAME']
    assert set(df.KEY.unique()) <= {'bonafide', 'spoof'}, f'unexpected KEY values in {fn}: {df.KEY.unique()[:5]}'
    flag = 'OK' if len(df) == EXPECTED_ROWS[split] else f'expected {EXPECTED_ROWS[split]}'
    print(f'    {split}: {len(df)} files ({flag}), {(df.KEY == "bonafide").sum()} bona fide, '
          f'{(df.KEY == "spoof").sum()} spoof, {df.SPEAKER_ID.nunique()} speakers')
    # key file in the official evaluation-package format (for Smitha's harness)
    save_tsv(df[['filename', 'KEY']].rename(columns={'KEY': 'cm-label'}),
             os.path.join(OUT_DIR, 'keys', f'{split}_cm_keys.tsv'))
    proto[split] = df
shutil.copy(find_proto(PROTO_FILES['eval']), os.path.join(OUT_DIR, 'keys'))  # full eval protocol, for the slices
print('Protocols loaded. Key files saved locally.')

--- ASVspoof 5 PROTOCOL CHECK ---
[EXISTS] ASVspoof5.train.tsv
    train: 182357 files (OK), 18797 bona fide, 163560 spoof, 400 speakers
[EXISTS] ASVspoof5.dev.track_1.tsv
    dev: 140950 files (OK), 31334 bona fide, 109616 spoof, 785 speakers
[EXISTS] ASVspoof5.eval.track_1.tsv
    eval: 680774 files (OK), 138688 bona fide, 542086 spoof, 737 speakers
Protocols loaded. Key files saved locally.


In [ ]:
# ===== CELL 4: Score the DEV set (3 tars, ~20 GB). Rerun if it stops; finished tars are skipped =====
dev_ids = set(proto['dev'].filename)
for t in DEV_TARS:
    run_tar_job('dev', t, dev_ids)

dev_scores = merge_parts('dev', DEV_TARS, dev_ids, 'aasist_dev.tsv')
res = metrics(dev_scores, proto['dev'], 'dev_full')
first2000 = set(proto['dev'].filename.iloc[:2000])
metrics(dev_scores[dev_scores.filename.isin(first2000)], proto['dev'], 'dev_first2000')
print('\nReproduction check: the official repo reports ~15.2% EER, measured on only the first 2000 dev files.')
print('dev_first2000 should be close to that; dev_full will be somewhat different, which is expected.')
if res and res['EER_%'] > 50:
    print('!! EER above 50% means the scores are flipped (real/fake swapped). Stop and check.')

In [ ]:
# ===== CELL 5: Score the EVAL set (10 tars, ~94 GB, one at a time). Rerun if it stops =====
eval_ids = set(proto['eval'].filename)
for t in EVAL_TARS:
    run_tar_job('eval', t, eval_ids)

eval_scores = merge_parts('eval', EVAL_TARS, eval_ids, 'aasist_eval.tsv')
metrics(eval_scores, proto['eval'], 'eval_pooled')
print('Pooled eval only. The per-shift slices are built by the evaluation harness from aasist_eval.tsv.')

In [ ]:
# ===== CELL 6: DFADD + VCTK bona fide: prepare the test set (16 kHz FLAC) and score it =====
DFADD_ZIPS = {'GradTTS': 'DATASET_GradTTS.zip', 'NaturalSpeech2': 'DATASET_NaturalSpeech2.zip',
              'StyleTTS2': 'DATASET_StyleTTS2.zip', 'MatchaTTS': 'DATASET_MatchaTTS.zip',
              'PflowTTS': 'DATASET_PflowTTS.zip', 'bonafide': 'DATASET_VCTK_BONAFIDE.zip'}
DFADD_DIR = os.path.join(LOCAL_DATA_DIR, 'dfadd_test_16k')
DFADD_TAR_PATH = os.path.join(OUT_DIR, 'dfadd', 'dfadd_test_16k.tar')
PREP_LOG = os.path.join(OUT_DIR, 'dfadd', 'dfadd_prep_log.txt')    # how each zip's test files were chosen
SUMMARY = os.path.join(OUT_DIR, 'dfadd', 'dfadd_summary.txt')      # counts, metrics, bona fide medians (for the report)
TEST_FOLDER = re.compile(r'(^|/)(test|testing)(/|_|$)', re.I)
TEST_SPEAKERS = re.compile(r'p22[78]')
EXPECTED_TEST_SPEAKERS = ['p227', 'p228']   # DFADD paper: test split = speakers p227 and p228
os.makedirs(DFADD_DIR, exist_ok=True)


def speakers_in(names):
    return sorted({m.group(0) for n in names for m in [re.search(r'p\d{3}', n)] if m})


if os.path.exists(DFADD_TAR_PATH) and not find_audio(DFADD_DIR):
    extract_tar(DFADD_TAR_PATH, LOCAL_DATA_DIR)          # already prepared in an earlier session

for gen, zname in DFADD_ZIPS.items():
    flag = os.path.join(DFADD_DIR, f'.{gen}.done')
    if os.path.exists(flag):
        print(f'[skip] {gen} already prepared')
        continue
    zpath = download(f'{DFADD_URL}/{zname}', os.path.join(WORK_DIR, zname))
    with zipfile.ZipFile(zpath) as z:
        audio = [n for n in z.namelist() if n.lower().endswith(('.wav', '.flac'))]
        by_folder = [n for n in audio if TEST_FOLDER.search(n)]
        by_speaker = [n for n in audio if TEST_SPEAKERS.search(n)]
        test = by_folder or by_speaker
        top = sorted({'/'.join(n.split('/')[:2]) for n in audio})[:15]
        lines = [f'{gen}: {len(audio)} audio files in zip',
                 f'  files in a test folder: {len(by_folder)} (speakers {speakers_in(by_folder)})',
                 f'  files from p227/p228:   {len(by_speaker)}',
                 f'  used: {"test folder" if by_folder else "speakers p227/p228"} -> {len(test)} files, speakers {speakers_in(test)}',
                 f'  both rules pick the same files: {set(by_folder) == set(by_speaker)}']
        if speakers_in(test) != EXPECTED_TEST_SPEAKERS:
            lines.append(f'  WARNING: expected test speakers {EXPECTED_TEST_SPEAKERS}. Check before using DFADD results.')
        print('\n'.join(lines))
        open(PREP_LOG, 'a').write('\n'.join(lines) + '\n')
        assert test, f'No test files found in {zname}. Folders inside: {top}'
        for n in tqdm(test, leave=False):
            x, sr = sf.read(io.BytesIO(z.read(n)), dtype='float32')
            if x.ndim > 1:
                x = x.mean(axis=1)
            if sr != 16000:
                g = math.gcd(sr, 16000)
                x = resample_poly(x, 16000 // g, sr // g).astype('float32')
            file_id = gen + '__' + re.sub(r'[^A-Za-z0-9_.-]', '_', os.path.splitext(n)[0])  # unique across generators
            sf.write(os.path.join(DFADD_DIR, file_id + '.flac'), np.clip(x, -1, 1), 16000, subtype='PCM_16')
    os.remove(zpath)
    open(flag, 'w').close()

dfadd_audio = find_audio(DFADD_DIR)
rows = []
for fid in sorted(dfadd_audio):
    gen = fid.split('__')[0]
    spk = re.search(r'p\d{3}', fid)
    rows.append({'SPEAKER_ID': spk.group(0) if spk else '-', 'filename': fid,
                 'GENERATOR': 'bonafide' if gen == 'bonafide' else gen,
                 'KEY': 'bonafide' if gen == 'bonafide' else 'spoof'})
dfadd_proto = pd.DataFrame(rows)
save_tsv(dfadd_proto, os.path.join(OUT_DIR, 'dfadd', 'dfadd_test_protocol.tsv'))

summary = ['DFADD test set: file counts per generator and speaker',
           pd.crosstab(dfadd_proto.GENERATOR, dfadd_proto.SPEAKER_ID, margins=True, margins_name='total').to_string(), '']
found = sorted(s for s in dfadd_proto.SPEAKER_ID.unique() if s != '-')
summary.append(f'Speakers: {found}  ' + ('(matches the DFADD test split)' if found == EXPECTED_TEST_SPEAKERS
                                          else f'!! expected {EXPECTED_TEST_SPEAKERS}'))

dfadd_scores = score(dfadd_proto.filename.tolist(), [dfadd_audio[f] for f in dfadd_proto.filename])
save_tsv(dfadd_scores, os.path.join(OUT_DIR, 'scores', 'aasist_dfadd.tsv'))

summary += ['', 'AASIST metrics (bona fide = all VCTK test files vs. each generator)']
bona = dfadd_proto[dfadd_proto.KEY == 'bonafide']
for gen in sorted(g for g in dfadd_proto.GENERATOR.unique() if g != 'bonafide'):
    subset = pd.concat([bona, dfadd_proto[dfadd_proto.GENERATOR == gen]])
    summary.append(str(metrics(dfadd_scores[dfadd_scores.filename.isin(subset.filename)], subset, f'dfadd_{gen}')))
summary.append(str(metrics(dfadd_scores, dfadd_proto, 'dfadd_pooled')))

# Bona fide check: DFADD's real speech is VCTK, not LibriVox, so compare real-speech scores separately
dev_file = os.path.join(OUT_DIR, 'scores', 'aasist_dev.tsv')
dev_all = pd.read_csv(dev_file, sep='\t', dtype={'filename': str})
dev_bona = dev_all[dev_all.filename.isin(proto['dev'][proto['dev'].KEY == 'bonafide'].filename)]['cm-score']
dfadd_bona = dfadd_scores[dfadd_scores.filename.isin(bona.filename)]['cm-score']
summary += ['', 'Bona fide (real speech) scores, higher = more real:',
            f'  DFADD / VCTK:     median {dfadd_bona.median():.2f}  (n={len(dfadd_bona)})',
            f'  ASVspoof 5 dev:   median {dev_bona.median():.2f}  (n={len(dev_bona)})',
            '  A large gap means AASIST rejects real VCTK speech, not just that it misses DFADD spoofs.']

print('\n'.join(summary))
open(SUMMARY, 'w').write('\n'.join(summary) + '\n')
print('\nSaved for the report:', SUMMARY)

if not os.path.exists(DFADD_TAR_PATH):
    with tarfile.open(DFADD_TAR_PATH, 'w') as archive:
        archive.add(DFADD_DIR, arcname=os.path.basename(DFADD_DIR))
    print('Prepared DFADD test audio saved locally:', DFADD_TAR_PATH)

In [ ]:
# ===== CELL 7: 10% held-out training speakers (the in-distribution reference) + AASIST scores on them =====
# The split is made ONCE and then reused, so everyone uses the same speakers.
split_file = os.path.join(OUT_DIR, 'splits', 'train_heldout_speakers.txt')
if os.path.exists(split_file):
    heldout = open(split_file).read().split()
    print(f'Using existing split: {len(heldout)} held-out speakers')
else:
    speakers = proto['train'].drop_duplicates('SPEAKER_ID')[['SPEAKER_ID', 'SPEAKER_GENDER']].sort_values('SPEAKER_ID')
    rng = np.random.RandomState(258)
    heldout = []
    for _, grp in speakers.groupby('SPEAKER_GENDER'):          # 10% of each gender
        ids = grp.SPEAKER_ID.to_numpy()
        heldout += list(rng.permutation(ids)[:max(1, round(0.10 * len(ids)))])
    heldout = sorted(heldout)
    open(split_file, 'w').write('\n'.join(heldout) + '\n')
    kept = sorted(set(speakers.SPEAKER_ID) - set(heldout))
    open(os.path.join(OUT_DIR, 'splits', 'train_kept_speakers.txt'), 'w').write('\n'.join(kept) + '\n')
    print(f'Created split: {len(heldout)} held-out speakers, {len(kept)} kept for training')

ref = proto['train'][proto['train'].SPEAKER_ID.isin(heldout)]
save_tsv(ref[['SPEAKER_ID', 'filename', 'ATTACK_LABEL', 'KEY']],
         os.path.join(OUT_DIR, 'splits', 'train_heldout_files.tsv'))
print(f'Held-out reference: {len(ref)} files ({(ref.KEY == "bonafide").sum()} bona fide)')

ref_ids = set(ref.filename)
for t in TRAIN_TARS:
    run_tar_job('trainref', t, ref_ids)
ref_scores = merge_parts('trainref', TRAIN_TARS, ref_ids, 'aasist_train_heldout_ref.tsv')
metrics(ref_scores, ref, 'train_heldout_ref')
print('Note: the released AASIST was trained on ALL training speakers, so this reference is optimistic for AASIST.')

In [ ]:
# # ===== CELL 8: Summary + README for the team =====
# allm = json.load(open(METRICS_FILE)) if os.path.exists(METRICS_FILE) else {}
# print(pd.DataFrame(allm.values()).to_string(index=False) if allm else 'No metrics yet.')

# readme = """AASIST (official ASVspoof 5 checkpoint) outputs - Shriya

# scores/aasist_dev.tsv, aasist_eval.tsv, aasist_dfadd.tsv, aasist_train_heldout_ref.tsv
#     filename <tab> cm-score <tab> logit_spoof <tab> logit_bonafide
#     cm-score = bona fide logit (official convention, higher = more likely real).
#     Readable by the official evaluation package: python evaluation.py --m t1 --cm <scores> --cm_keys <keys>
# keys/{train,dev,eval}_cm_keys.tsv    filename <tab> cm-label, official format
# keys/ASVspoof5.eval.track_1.tsv      full eval protocol (codec, CODEC_SEED, attack) for building slices
# dfadd/dfadd_test_protocol.tsv        SPEAKER_ID, filename, GENERATOR, KEY for the DFADD test set
# dfadd/dfadd_test_16k.tar             prepared DFADD test audio, 16 kHz mono FLAC (bona fide = VCTK)
# splits/train_heldout_speakers.txt    the 10% held-out training speakers (seed 258, stratified by gender)
# splits/train_kept_speakers.txt       the other 90% (use these to train the SSL head)
# metrics/aasist_metrics.json          official metrics (minDCF, EER, CLLR, actDCF) per set
# Model input: first 64,600 samples, short clips repeated (same as the official AASIST code).
# """
# open(os.path.join(OUT_DIR, 'README.txt'), 'w').write(readme)

# print('\nFiles in', OUT_DIR)
# for dirpath, _, files in sorted(os.walk(OUT_DIR)):
#     if os.path.basename(dirpath) == 'parts':
#         continue
#     for fn in sorted(files):
#         p = os.path.join(dirpath, fn)
#         print(f'  {os.path.relpath(p, OUT_DIR):45s} {os.path.getsize(p) / 1e6:9.1f} MB')


# ===== CELL 8: Summary + README for the team =====
allm = json.load(open(METRICS_FILE)) if os.path.exists(METRICS_FILE) else {}
print(pd.DataFrame(allm.values()).to_string(index=False) if allm else 'No metrics yet.')

dfadd_done = os.path.exists(os.path.join(OUT_DIR, 'scores', 'aasist_dfadd.tsv'))
pending = '' if dfadd_done else '   [PENDING: Cell 6 not run yet]'
readme = f"""AASIST (official ASVspoof 5 checkpoint) outputs - Shriya

scores/aasist_dev.tsv, aasist_eval.tsv, aasist_train_heldout_ref.tsv, aasist_dfadd.tsv{pending}
    filename <tab> cm-score <tab> logit_spoof <tab> logit_bonafide
    cm-score = bona fide logit (official convention, higher = more likely real).
    Readable by the official evaluation package: python evaluation.py --m t1 --cm <scores> --cm_keys <keys>
keys/{{train,dev,eval}}_cm_keys.tsv    filename <tab> cm-label, official format
keys/ASVspoof5.eval.track_1.tsv      full eval protocol (codec, CODEC_SEED, attack) for building slices
splits/train_heldout_speakers.txt    the 10% held-out training speakers (seed 258, stratified by gender)
splits/train_kept_speakers.txt       the other 90% (use these to train the SSL head)
splits/train_heldout_files.tsv       every training file from the held-out speakers
metrics/aasist_metrics.json          official metrics (minDCF, EER, CLLR, actDCF) per set
ssl_detector/cache/                    XLS-R train/dev/eval per-layer feature caches
ssl_detector/models/                   three seed-specific XLS-R head checkpoints
ssl_detector/scores/                   XLS-R dev/eval raw scores
metrics/ssl_calibration/          AASIST/XLS-R ECE, reliability and risk-coverage
scores/ssl_calibrated/            calibrated AASIST/XLS-R dev/eval scores
dfadd/dfadd_test_protocol.tsv        SPEAKER_ID, filename, GENERATOR, KEY for the DFADD test set{pending}
dfadd/dfadd_test_16k.tar             prepared DFADD test audio, 16 kHz mono FLAC (bona fide = VCTK){pending}
dfadd/dfadd_summary.txt              counts per generator/speaker, metrics, bona fide medians{pending}
dfadd/dfadd_prep_log.txt             how each DFADD zip's test files were selected{pending}

Notes
- Model input: first 64,600 samples, short clips repeated (same as the official AASIST code).
- Reproduction: dev_full EER 15.20% matches the official repo's 15.2%. Ignore dev_first2000; the official
  code's first 2000 files come from an older metadata file, so it is not the same subset.
- train_heldout_ref (0.05% EER) is NOT held out for AASIST: the released checkpoint was trained on all
  training speakers. Do not use it as AASIST's in-distribution reference. It is the right reference for
  the SSL head, which never sees these speakers.
- Scores are raw, uncalibrated logits, so actDCF and CLLR look poor everywhere. Report minDCF and EER;
  report actDCF only after temperature scaling is fitted on dev.
"""
open(os.path.join(OUT_DIR, 'README.txt'), 'w').write(readme)

print('\nFiles in', OUT_DIR)
for dirpath, _, files in sorted(os.walk(OUT_DIR)):
    if os.path.basename(dirpath) == 'parts':
        continue
    for fn in sorted(files):
        p = os.path.join(dirpath, fn)
        print(f'  {os.path.relpath(p, OUT_DIR):45s} {os.path.getsize(p) / 1e6:9.1f} MB')

## Zahira's XLS-R, Calibration, and Selective Prediction

For Zahira's work, run only the notebook setup/helper/protocol cells (Cells 1?3), then the new cells below in order. **You do not need to run the AASIST dev/eval/DFADD scoring cells** to train and evaluate XLS-R. The first cell creates or reuses the shared 90% train-speaker split. Install the two missing packages, cache train/dev features, train the 3-seed head, then cache/score eval and run dev-fitted calibration.

The cache is resumable by shard. It writes about 100 GB of float16 features for train/dev/eval combined, plus the XLS-R model download; check free disk before starting. The audio archives are downloaded one at a time and deleted after caching. A GPU is strongly recommended. Evaluation labels are used only for the final report, not training or temperature fitting.

In [6]:
# Create/reuse the project-wide, gender-stratified speaker split without running AASIST scoring.
from pathlib import Path

SSL_OUTPUT_DIR = os.path.join(OUT_DIR, 'ssl_detector')
SSL_CACHE_DIR = os.path.join(SSL_OUTPUT_DIR, 'cache')
SSL_MODEL_DIR = os.path.join(SSL_OUTPUT_DIR, 'models')
SSL_SCORE_DIR = os.path.join(SSL_OUTPUT_DIR, 'scores')
SSL_METRICS_DIR = os.path.join(SSL_OUTPUT_DIR, 'metrics')
for folder in [SSL_CACHE_DIR, SSL_MODEL_DIR, SSL_SCORE_DIR, SSL_METRICS_DIR]:
    os.makedirs(folder, exist_ok=True)

heldout_path = os.path.join(OUT_DIR, 'splits', 'train_heldout_speakers.txt')
kept_path = os.path.join(OUT_DIR, 'splits', 'train_kept_speakers.txt')
os.makedirs(os.path.dirname(heldout_path), exist_ok=True)
if os.path.exists(heldout_path) and os.path.exists(kept_path):
    ssl_heldout = set(open(heldout_path, encoding='utf-8').read().split())
    ssl_kept = set(open(kept_path, encoding='utf-8').read().split())
    print(f'Reusing shared split: {len(ssl_kept)} train speakers, {len(ssl_heldout)} held out')
else:
    ssl_speakers = (proto['train'][['SPEAKER_ID', 'SPEAKER_GENDER']]
                       .drop_duplicates().sort_values('SPEAKER_ID'))
    ssl_rng = np.random.RandomState(258)
    ssl_heldout_list = []
    for _, group in ssl_speakers.groupby('SPEAKER_GENDER'):
        speaker_ids = group.SPEAKER_ID.to_numpy()
        ssl_heldout_list.extend(
            ssl_rng.permutation(speaker_ids)[:max(1, round(0.10 * len(speaker_ids)))]
        )
    ssl_heldout = set(sorted(ssl_heldout_list))
    ssl_kept = set(ssl_speakers.SPEAKER_ID) - ssl_heldout
    Path(heldout_path).write_text('\n'.join(sorted(ssl_heldout)) + '\n', encoding='utf-8')
    Path(kept_path).write_text('\n'.join(sorted(ssl_kept)) + '\n', encoding='utf-8')
    print(f'Created shared split: {len(ssl_kept)} train speakers, {len(ssl_heldout)} held out')

print('XLS-R training examples:', int(proto['train'].SPEAKER_ID.isin(ssl_kept).sum()))

Reusing shared split: 360 train speakers, 40 held out
XLS-R training examples: 163774


In [5]:
%pip install "transformers>=4.45,<5" matplotlib

Note: you may need to restart the kernel to use updated packages.


In [6]:
# Frozen XLS-R per-layer mean/std feature extraction; caches resume safely by completed tar shard.
import io
from transformers import AutoFeatureExtractor, AutoModel

SSL_MODEL_NAME = 'facebook/wav2vec2-xls-r-300m'
SSL_BATCH_SIZE = 2
SSL_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SSL_EXTRACTOR = AutoFeatureExtractor.from_pretrained(SSL_MODEL_NAME)
SSL_ENCODER = AutoModel.from_pretrained(
    SSL_MODEL_NAME, output_hidden_states=True
).to(SSL_DEVICE).eval()
for parameter in SSL_ENCODER.parameters():
    parameter.requires_grad_(False)
SSL_NUM_LAYERS = int(SSL_ENCODER.config.num_hidden_layers) + 1
SSL_FEATURE_DIM = int(SSL_ENCODER.config.hidden_size) * 2
print(f'Frozen XLS-R ready on {SSL_DEVICE}: {SSL_NUM_LAYERS} layers, {SSL_FEATURE_DIM} statistics per layer')


def ssl_audio_waveform(audio_bytes, display_name):
    try:
        waveform, sample_rate = sf.read(io.BytesIO(audio_bytes), dtype='float32')
    except Exception as exc:
        raise RuntimeError(f'Cannot decode {display_name}: {exc}') from exc
    if sample_rate != 16000:
        raise ValueError(f'{display_name} is {sample_rate} Hz; ASVspoof 5 audio must be 16000 Hz')
    if waveform.ndim > 1:
        waveform = waveform.mean(axis=1)
    if len(waveform) == 0:
        waveform = np.zeros(CUT, dtype=np.float32)
    elif len(waveform) < CUT:
        waveform = np.tile(waveform, CUT // len(waveform) + 1)[:CUT]
    else:
        waveform = waveform[:CUT]
    return np.ascontiguousarray(waveform, dtype=np.float32)


@torch.inference_mode()
def ssl_encode_batch(waveforms):
    inputs = SSL_EXTRACTOR(
        waveforms, sampling_rate=16000, return_tensors='pt', padding=True
    )
    outputs = SSL_ENCODER(inputs['input_values'].to(SSL_DEVICE), output_hidden_states=True)
    if outputs.hidden_states is None:
        raise RuntimeError('XLS-R did not return per-layer hidden states')
    stats = []
    for layer in outputs.hidden_states:
        stats.append(torch.cat((layer.mean(dim=1), layer.std(dim=1, unbiased=False)), dim=-1))
    return torch.stack(stats, dim=1).to(device='cpu', dtype=torch.float16).numpy()


def ssl_protocol(split):
    frame = proto[split].copy()
    if split == 'train':
        frame = frame[frame.SPEAKER_ID.isin(ssl_kept)]
    return frame.reset_index(drop=True)


def ssl_cache_split(split, tar_names):
    frame = ssl_protocol(split)
    filenames = frame.filename.astype(str).tolist()
    feature_shape = (len(frame), SSL_NUM_LAYERS, SSL_FEATURE_DIM)
    final_features = os.path.join(SSL_CACHE_DIR, f'{split}.features.f16')
    final_metadata = os.path.join(SSL_CACHE_DIR, f'{split}.json')
    partial_features = os.path.join(SSL_CACHE_DIR, f'{split}.partial.f16')
    progress_path = os.path.join(SSL_CACHE_DIR, f'{split}.progress.json')
    if os.path.isfile(final_features) and os.path.isfile(final_metadata):
        print(f'[skip] complete {split} cache already exists')
        return

    needed_bytes = int(np.prod(feature_shape)) * np.dtype(np.float16).itemsize
    free_bytes = shutil.disk_usage(SSL_CACHE_DIR).free
    if not os.path.exists(progress_path) and needed_bytes + 10_000_000_000 > free_bytes:
        raise OSError(
            f'{split} feature cache needs {needed_bytes / 1e9:.1f} GB plus 10 GB headroom; '
            f'only {free_bytes / 1e9:.1f} GB is free.'
        )
    if os.path.exists(progress_path):
        progress = json.load(open(progress_path, encoding='utf-8'))
        if (progress['filenames'] != filenames or progress['model_name'] != SSL_MODEL_NAME
                or progress['shape'] != list(feature_shape)):
            raise ValueError(f'Partial {split} cache metadata does not match the current protocol/model.')
        completed_tars = set(progress['completed_tars'])
        completed_indices = set(progress['completed_indices'])
        feature_map = np.memmap(partial_features, mode='r+', dtype=np.float16, shape=feature_shape)
        print(f'Resuming {split}: {len(completed_indices):,}/{len(filenames):,} files cached')
    else:
        completed_tars, completed_indices = set(), set()
        feature_map = np.memmap(partial_features, mode='w+', dtype=np.float16, shape=feature_shape)
        progress = {
            'filenames': filenames, 'model_name': SSL_MODEL_NAME,
            'shape': list(feature_shape), 'completed_tars': [], 'completed_indices': []
        }
        with open(progress_path, 'w', encoding='utf-8') as stream:
            json.dump(progress, stream)

    index_by_id = {filename: index for index, filename in enumerate(filenames)}
    seen_indices = set(completed_indices)
    try:
        for tar_name in tar_names:
            if tar_name in completed_tars:
                print(f'[skip] already cached {tar_name}')
                continue
            tar_path = os.path.join(WORK_DIR, tar_name)
            download(f'{ZENODO_URL}/{tar_name}', tar_path)
            shard_indices = set()
            batch_indices, batch_waveforms = [], []

            def flush_ssl_batch():
                if not batch_indices:
                    return
                encoded = ssl_encode_batch(batch_waveforms)
                if encoded.shape[1:] != feature_shape[1:]:
                    raise RuntimeError(f'Unexpected XLS-R feature shape {encoded.shape[1:]}')
                feature_map[batch_indices] = encoded
                batch_indices.clear()
                batch_waveforms.clear()

            try:
                with tarfile.open(tar_path, mode='r:*') as archive:
                    for member in archive:
                        if not member.isfile() or not member.name.lower().endswith(('.flac', '.wav')):
                            continue
                        file_id = os.path.splitext(os.path.basename(member.name))[0]
                        index = index_by_id.get(file_id)
                        if index is None:
                            continue
                        if index in seen_indices or index in shard_indices:
                            raise ValueError(f'Duplicate protocol audio ID in {tar_name}: {file_id}')
                        source = archive.extractfile(member)
                        if source is None:
                            raise RuntimeError(f'Cannot read {member.name} from {tar_name}')
                        with source:
                            batch_waveforms.append(ssl_audio_waveform(source.read(), f'{tar_name}:{member.name}'))
                        batch_indices.append(index)
                        shard_indices.add(index)
                        if len(batch_indices) >= SSL_BATCH_SIZE:
                            flush_ssl_batch()
                    flush_ssl_batch()
            except (tarfile.TarError, EOFError) as exc:
                raise RuntimeError(
                    f'{tar_name} is incomplete or damaged. Remove that tar from {WORK_DIR} and rerun this cell.'
                ) from exc
            if not shard_indices:
                raise RuntimeError(f'No {split} protocol audio found in {tar_name}; verify the archive/split names.')
            seen_indices.update(shard_indices)
            completed_indices.update(shard_indices)
            completed_tars.add(tar_name)
            feature_map.flush()
            progress['completed_tars'] = sorted(completed_tars)
            progress['completed_indices'] = sorted(completed_indices)
            temporary_progress = progress_path + '.tmp'
            with open(temporary_progress, 'w', encoding='utf-8') as stream:
                json.dump(progress, stream)
            os.replace(temporary_progress, progress_path)
            os.remove(tar_path)
            print(f'{split}: cached {len(completed_indices):,}/{len(filenames):,} after {tar_name}')

        missing = sorted(set(range(len(filenames))) - completed_indices)
        if missing:
            missing_names = [filenames[index] for index in missing[:10]]
            raise RuntimeError(
                f'{split} cache is missing {len(missing):,} audio files. First IDs: {missing_names}. '
                'Check all listed archives are complete before training.'
            )
        feature_map.flush()
        del feature_map
        os.replace(partial_features, final_features)
        metadata = {
            'split': split, 'shape': list(feature_shape), 'dtype': 'float16',
            'model_name': SSL_MODEL_NAME, 'filenames': filenames,
            'speakers': frame.SPEAKER_ID.astype(str).tolist(), 'keys': frame.KEY.astype(str).tolist(),
        }
        temporary_metadata = final_metadata + '.tmp'
        with open(temporary_metadata, 'w', encoding='utf-8') as stream:
            json.dump(metadata, stream)
        os.replace(temporary_metadata, final_metadata)
        os.remove(progress_path)
        print(f'Saved complete {split} feature cache: {final_features}')
    except Exception:
        feature_map.flush()
        del feature_map
        raise


def ssl_load_cache(split):
    feature_path = os.path.join(SSL_CACHE_DIR, f'{split}.features.f16')
    metadata_path = os.path.join(SSL_CACHE_DIR, f'{split}.json')
    if not os.path.isfile(feature_path) or not os.path.isfile(metadata_path):
        raise FileNotFoundError(f'Run the Zahira cache cell for {split} first.')
    metadata = json.load(open(metadata_path, encoding='utf-8'))
    features = np.memmap(
        feature_path, mode='r', dtype=np.float16, shape=tuple(metadata['shape'])
    )
    return features, metadata

preprocessor_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

c:\Users\ghazz\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ghazz\.cache\huggingface\hub\models--facebook--wav2vec2-xls-r-300m. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Frozen XLS-R ready on cuda: 25 layers, 2048 statistics per layer


In [7]:
# Heavy step: caches only kept-speaker train audio and the full dev set; rerun to resume by shard.
ssl_cache_split('train', TRAIN_TARS)
ssl_cache_split('dev', DEV_TARS)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

  flac_T_aa.tar: downloaded (7174 MB)          
train: cached 32,780/163,774 after flac_T_aa.tar
  flac_T_ab.tar: downloaded (7142 MB)          
train: cached 65,621/163,774 after flac_T_ab.tar
  flac_T_ac.tar: downloaded (7149 MB)          
train: cached 98,255/163,774 after flac_T_ac.tar
  flac_T_ad.tar: downloaded (7161 MB)          
train: cached 131,126/163,774 after flac_T_ad.tar
  flac_T_ae.tar: downloaded (7135 MB)          
train: cached 163,774/163,774 after flac_T_ae.tar
Saved complete train feature cache: c:\Users\ghazz\Downloads\DeepLearningProj\outputs\shriya_aasist\ssl_detector\cache\train.features.f16
  flac_D_aa.tar: downloaded (6339 MB)          
dev: cached 47,400/140,950 after flac_D_aa.tar
  flac_D_ab.tar: downloaded (6333 MB)          
dev: cached 94,800/140,950 after flac_D_ab.tar
  flac_D_ac.tar: downloaded (6457 MB)          
dev: cached 140,950/140,950 after flac_D_ac.tar
Saved complete dev feature cache: c:\Users\ghazz\Downloads\DeepLearningProj\outputs\shriy

In [8]:
# Train three lightweight layer-weighted heads; dev loss selects the checkpoint/epoch.
class SSLFeatureDataset(Dataset):
    def __init__(self, feature_map, labels):
        self.feature_map = feature_map
        self.labels = np.asarray([1 if key == 'bonafide' else 0 for key in labels], dtype=np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        values = np.asarray(self.feature_map[index], dtype=np.float32)
        return torch.from_numpy(values), torch.tensor(self.labels[index], dtype=torch.long)


class SSLLayerWeightedHead(torch.nn.Module):
    def __init__(self, num_layers, feature_dim):
        super().__init__()
        self.layer_weights = torch.nn.Parameter(torch.zeros(num_layers))
        self.classifier = torch.nn.Sequential(
            torch.nn.LayerNorm(feature_dim), torch.nn.Linear(feature_dim, 256),
            torch.nn.GELU(), torch.nn.Dropout(0.2), torch.nn.Linear(256, 2)
        )

    def forward(self, features):
        weights = torch.softmax(self.layer_weights, dim=0)
        pooled = (features * weights[None, :, None]).sum(dim=1)
        return self.classifier(pooled)


def ssl_predict(model, feature_map, batch_size=256):
    loader = DataLoader(
        SSLFeatureDataset(feature_map, ['spoof'] * len(feature_map)),
        batch_size=batch_size, shuffle=False, num_workers=0
    )
    model.eval()
    outputs = []
    with torch.inference_mode():
        for values, _ in loader:
            outputs.append(model(values.to(SSL_DEVICE)).cpu().numpy())
    return np.concatenate(outputs, axis=0)


def ssl_scores_frame(logits, filenames):
    return pd.DataFrame({
        'filename': filenames,
        'cm-score': logits[:, 1] - logits[:, 0],
        'logit_spoof': logits[:, 0],
        'logit_bonafide': logits[:, 1],
    })


train_features, train_meta = ssl_load_cache('train')
dev_features, dev_meta = ssl_load_cache('dev')
if train_meta['model_name'] != dev_meta['model_name'] or train_meta['shape'][1:] != dev_meta['shape'][1:]:
    raise ValueError('Train and dev feature caches must use the same XLS-R model and dimensions.')

SSL_SEEDS = (258, 259, 260)
SSL_EPOCHS = 10
SSL_HEAD_BATCH_SIZE = 256
seed_dev_logits = []
os.makedirs(SSL_MODEL_DIR, exist_ok=True)
os.makedirs(SSL_SCORE_DIR, exist_ok=True)

for seed in SSL_SEEDS:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if SSL_DEVICE.type == 'cuda':
        torch.cuda.manual_seed_all(seed)
    head = SSLLayerWeightedHead(train_features.shape[1], train_features.shape[2]).to(SSL_DEVICE)
    train_data = SSLFeatureDataset(train_features, train_meta['keys'])
    dev_data = SSLFeatureDataset(dev_features, dev_meta['keys'])
    train_loader = DataLoader(
        train_data, batch_size=SSL_HEAD_BATCH_SIZE, shuffle=True,
        generator=torch.Generator().manual_seed(seed), num_workers=0,
        pin_memory=SSL_DEVICE.type == 'cuda'
    )
    dev_loader = DataLoader(dev_data, batch_size=SSL_HEAD_BATCH_SIZE, shuffle=False, num_workers=0)
    optimizer = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
    best_loss, best_state, best_epoch, stale = float('inf'), None, 0, 0
    history = []
    for epoch in range(1, SSL_EPOCHS + 1):
        head.train()
        for values, labels in train_loader:
            values, labels = values.to(SSL_DEVICE), labels.to(SSL_DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = torch.nn.functional.cross_entropy(head(values), labels)
            loss.backward()
            optimizer.step()
        head.eval()
        dev_loss_sum, dev_count = 0.0, 0
        with torch.inference_mode():
            for values, labels in dev_loader:
                values, labels = values.to(SSL_DEVICE), labels.to(SSL_DEVICE)
                batch_loss = torch.nn.functional.cross_entropy(head(values), labels)
                dev_loss_sum += float(batch_loss) * len(labels)
                dev_count += len(labels)
        dev_loss = dev_loss_sum / dev_count
        history.append({'epoch': epoch, 'dev_loss': dev_loss})
        print(f'seed={seed} epoch={epoch}/{SSL_EPOCHS} dev_loss={dev_loss:.5f}')
        if dev_loss < best_loss:
            best_loss = dev_loss
            best_epoch = epoch
            best_state = {key: value.detach().cpu().clone() for key, value in head.state_dict().items()}
            stale = 0
        else:
            stale += 1
            if stale >= 3:
                break
    if best_state is None:
        raise RuntimeError(f'No best checkpoint was produced for seed {seed}')
    head.load_state_dict(best_state)
    checkpoint_path = os.path.join(SSL_MODEL_DIR, f'xlsr_seed_{seed}.pt')
    torch.save({
        'seed': seed, 'model_name': SSL_MODEL_NAME,
        'num_layers': train_features.shape[1], 'feature_dim': train_features.shape[2],
        'best_epoch': best_epoch, 'best_dev_loss': best_loss, 'state_dict': best_state
    }, checkpoint_path)
    seed_logits = ssl_predict(head, dev_features, SSL_HEAD_BATCH_SIZE)
    seed_dev_logits.append(seed_logits)
    save_tsv(ssl_scores_frame(seed_logits, dev_meta['filenames']),
             os.path.join(SSL_SCORE_DIR, f'xlsr_dev_seed_{seed}.tsv'))
    with open(os.path.join(SSL_MODEL_DIR, f'xlsr_seed_{seed}_history.json'), 'w', encoding='utf-8') as stream:
        json.dump(history, stream, indent=2)
    print(f'Saved best seed {seed} (epoch {best_epoch})')

ensemble_dev_logits = np.mean(seed_dev_logits, axis=0)
save_tsv(ssl_scores_frame(ensemble_dev_logits, dev_meta['filenames']),
         os.path.join(SSL_SCORE_DIR, 'xlsr_dev.tsv'))
print('Saved ensemble dev scores; dev is also the only data used to fit temperature.')

seed=258 epoch=1/10 dev_loss=0.08729
seed=258 epoch=2/10 dev_loss=0.05594
seed=258 epoch=3/10 dev_loss=0.16994
seed=258 epoch=4/10 dev_loss=0.15568
seed=258 epoch=5/10 dev_loss=0.22647
Saved best seed 258 (epoch 2)
seed=259 epoch=1/10 dev_loss=0.06214
seed=259 epoch=2/10 dev_loss=0.09012
seed=259 epoch=3/10 dev_loss=0.05993
seed=259 epoch=4/10 dev_loss=0.09531
seed=259 epoch=5/10 dev_loss=0.19019
seed=259 epoch=6/10 dev_loss=0.12833
Saved best seed 259 (epoch 3)
seed=260 epoch=1/10 dev_loss=0.06382
seed=260 epoch=2/10 dev_loss=0.11736
seed=260 epoch=3/10 dev_loss=0.03433
seed=260 epoch=4/10 dev_loss=0.27850
seed=260 epoch=5/10 dev_loss=0.14585
seed=260 epoch=6/10 dev_loss=0.11113
Saved best seed 260 (epoch 3)
Saved ensemble dev scores; dev is also the only data used to fit temperature.


In [9]:
# Heavy step: cache eval audio only after training/model selection is finished.
ssl_cache_split('eval', EVAL_TARS)

  flac_E_aa.tar: downloaded (8058 MB)          
eval: cached 68,188/680,774 after flac_E_aa.tar
  flac_E_ab.tar: downloaded (8067 MB)          
eval: cached 136,376/680,774 after flac_E_ab.tar
  flac_E_ac.tar: downloaded (8069 MB)          
eval: cached 204,564/680,774 after flac_E_ac.tar
  flac_E_ad.tar: downloaded (8074 MB)          
eval: cached 272,752/680,774 after flac_E_ad.tar
  flac_E_ae.tar: downloaded (8065 MB)          
eval: cached 340,940/680,774 after flac_E_ae.tar
  flac_E_af.tar: downloaded (8065 MB)          
eval: cached 409,128/680,774 after flac_E_af.tar
  flac_E_ag.tar: downloaded (8069 MB)          
eval: cached 477,316/680,774 after flac_E_ag.tar
  flac_E_ah.tar: downloaded (8063 MB)          
eval: cached 545,504/680,774 after flac_E_ah.tar
  flac_E_ai.tar: downloaded (8072 MB)          
eval: cached 613,692/680,774 after flac_E_ai.tar
  flac_E_aj.tar: downloaded (8223 MB)          
eval: cached 680,774/680,774 after flac_E_aj.tar
Saved complete eval feature cac

In [10]:
# Score eval using the frozen 3-seed ensemble, then fit temperature on dev only.
eval_features, eval_meta = ssl_load_cache('eval')
seed_eval_logits = []
for seed in SSL_SEEDS:
    checkpoint_path = os.path.join(SSL_MODEL_DIR, f'xlsr_seed_{seed}.pt')
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    if checkpoint['model_name'] != SSL_MODEL_NAME:
        raise ValueError(f'Checkpoint/model mismatch: {checkpoint_path}')
    head = SSLLayerWeightedHead(checkpoint['num_layers'], checkpoint['feature_dim']).to(SSL_DEVICE)
    head.load_state_dict(checkpoint['state_dict'])
    logits = ssl_predict(head, eval_features, SSL_HEAD_BATCH_SIZE)
    seed_eval_logits.append(logits)
    save_tsv(ssl_scores_frame(logits, eval_meta['filenames']),
             os.path.join(SSL_SCORE_DIR, f'xlsr_eval_seed_{seed}.tsv'))
ensemble_eval_logits = np.mean(seed_eval_logits, axis=0)
eval_score_frame = ssl_scores_frame(ensemble_eval_logits, eval_meta['filenames'])
save_tsv(eval_score_frame, os.path.join(SSL_SCORE_DIR, 'xlsr_eval.tsv'))

import matplotlib.pyplot as plt


def ssl_fit_temperature(logits, labels):
    scores = torch.tensor(logits, dtype=torch.float64)
    targets = torch.tensor(labels, dtype=torch.long)
    log_temperature = torch.zeros((), dtype=torch.float64, requires_grad=True)
    optimizer = torch.optim.LBFGS([log_temperature], lr=0.1, max_iter=100)
    def closure():
        optimizer.zero_grad()
        temperature = log_temperature.exp().clamp(1e-3, 1e3)
        loss = torch.nn.functional.cross_entropy(scores / temperature, targets)
        loss.backward()
        return loss
    optimizer.step(closure)
    return float(log_temperature.detach().exp().clamp(1e-3, 1e3))


def ssl_ece(probabilities, labels, bins=15):
    edges = np.linspace(0.0, 1.0, bins + 1)
    rows, ece = [], 0.0
    for index in range(bins):
        lo, hi = edges[index], edges[index + 1]
        mask = (probabilities >= lo) & (probabilities <= hi if index == bins - 1 else probabilities < hi)
        count = int(mask.sum())
        confidence = float(probabilities[mask].mean()) if count else np.nan
        accuracy = float(labels[mask].mean()) if count else np.nan
        if count:
            ece += count / len(labels) * abs(confidence - accuracy)
        rows.append({'bin_lower': lo, 'bin_upper': hi, 'count': count,
                     'confidence': confidence, 'accuracy': accuracy})
    return ece, pd.DataFrame(rows)


def ssl_selective(probabilities, labels):
    predictions = (probabilities >= 0.5).astype(np.int64)
    errors = (predictions != labels).astype(np.int64)
    confidence = np.maximum(probabilities, 1.0 - probabilities)
    order = np.argsort(-confidence, kind='stable')
    ordered_errors = errors[order]
    count = np.arange(1, len(labels) + 1)
    coverage = count / len(labels)
    risk = np.cumsum(ordered_errors) / count
    oracle_risk = np.cumsum(np.sort(errors)) / count
    report = {
        'aurc': float(np.trapezoid(np.r_[0.0, risk], np.r_[0.0, coverage])),
        'oracle_aurc': float(np.trapezoid(np.r_[0.0, oracle_risk], np.r_[0.0, coverage])),
    }
    report['excess_aurc'] = report['aurc'] - report['oracle_aurc']
    for pct in (80, 90):
        retained = int(np.ceil(pct / 100 * len(labels)))
        report[f'risk_at_{pct}_coverage'] = float(ordered_errors[:retained].mean())
    curve = pd.DataFrame({'coverage': coverage, 'risk': risk,
                          'confidence': confidence[order], 'error': ordered_errors})
    return report, curve


dev_labels = (np.asarray(dev_meta['keys']) == 'bonafide').astype(np.int64)
eval_labels = (np.asarray(eval_meta['keys']) == 'bonafide').astype(np.int64)
dev_logits = np.column_stack((
    pd.read_csv(os.path.join(SSL_SCORE_DIR, 'xlsr_dev.tsv'), sep='	').logit_spoof,
    pd.read_csv(os.path.join(SSL_SCORE_DIR, 'xlsr_dev.tsv'), sep='	').logit_bonafide,
))
eval_logits = np.column_stack((eval_score_frame.logit_spoof, eval_score_frame.logit_bonafide))
if len(dev_logits) != len(dev_labels) or len(eval_logits) != len(eval_labels):
    raise ValueError('XLS-R scores and protocol labels are not aligned.')
temperature = ssl_fit_temperature(dev_logits, dev_labels)
print(f'Dev-fitted temperature: {temperature:.5f}')

calibration_report = {'temperature_fitted_on': 'dev', 'temperature': temperature, 'splits': {}}
for split, logits, labels, metadata in (
    ('dev', dev_logits, dev_labels, dev_meta), ('eval', eval_logits, eval_labels, eval_meta)
):
    raw_prob = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]
    calibrated_logits = logits / temperature
    calibrated_prob = torch.softmax(torch.tensor(calibrated_logits), dim=1).numpy()[:, 1]
    raw_ece, _ = ssl_ece(raw_prob, labels)
    calibrated_ece, reliability = ssl_ece(calibrated_prob, labels)
    selective, risk_curve = ssl_selective(calibrated_prob, labels)
    result = {'n_files': len(labels), 'raw_ECE': raw_ece, 'calibrated_ECE': calibrated_ece,
              'selective_prediction': selective}
    score_frame = ssl_scores_frame(calibrated_logits, metadata['filenames'])
    score_frame['bonafide_probability'] = calibrated_prob
    save_tsv(score_frame, os.path.join(SSL_SCORE_DIR, f'xlsr_{split}_calibrated.tsv'))
    reliability.to_csv(os.path.join(SSL_METRICS_DIR, f'xlsr_{split}_reliability.tsv'), sep='	', index=False)
    risk_curve.to_csv(os.path.join(SSL_METRICS_DIR, f'xlsr_{split}_risk_coverage.tsv'), sep='	', index=False)

    fig, axis = plt.subplots(figsize=(6, 6))
    populated = reliability[reliability['count'] > 0]
    axis.plot([0, 1], [0, 1], '--', color='gray', label='Perfect calibration')
    axis.plot(populated.confidence, populated.accuracy, marker='o', label='XLS-R')
    axis.set(xlabel='Mean predicted bona fide probability', ylabel='Bona fide fraction',
             title=f'{split.upper()} reliability diagram', xlim=(0, 1), ylim=(0, 1))
    axis.grid(alpha=0.25); axis.legend(); fig.tight_layout()
    fig.savefig(os.path.join(SSL_OUTPUT_DIR, f'xlsr_{split}_reliability.png'), dpi=160)
    plt.close(fig)

    fig, axis = plt.subplots(figsize=(6, 5))
    axis.plot(risk_curve.coverage, risk_curve.risk)
    axis.set(xlabel='Coverage', ylabel='Error among retained files',
             title=f'{split.upper()} risk?coverage', xlim=(0, 1), ylim=(0, 1))
    axis.grid(alpha=0.25); fig.tight_layout()
    fig.savefig(os.path.join(SSL_OUTPUT_DIR, f'xlsr_{split}_risk_coverage.png'), dpi=160)
    plt.close(fig)

    if 'calculate_minDCF_EER_CLLR_actDCF' in globals():
        official = metrics(score_frame, pd.DataFrame({'filename': metadata['filenames'], 'KEY': metadata['keys']}),
                           f'xlsr_{split}_calibrated')
        result['official_metrics'] = official
    calibration_report['splits'][split] = result

report_path = os.path.join(SSL_METRICS_DIR, 'xlsr_calibration_and_selective.json')
with open(report_path, 'w', encoding='utf-8') as stream:
    json.dump(calibration_report, stream, indent=2)
print(json.dumps(calibration_report, indent=2))
print('Reliability, risk?coverage, calibrated scores, and metrics saved under', SSL_OUTPUT_DIR)

Dev-fitted temperature: 1.52230
{'set': 'xlsr_dev_calibrated', 'n_files': 140950, 'n_bonafide': 31334, 'minDCF': 0.0222, 'EER_%': 0.99, 'CLLR': 0.0493, 'actDCF': 0.0236}
{'set': 'xlsr_eval_calibrated', 'n_files': 680774, 'n_bonafide': 138688, 'minDCF': 0.2149, 'EER_%': 7.43, 'CLLR': 0.3937, 'actDCF': 0.2231}
{
  "temperature_fitted_on": "dev",
  "temperature": 1.522302054847988,
  "splits": {
    "dev": {
      "n_files": 140950,
      "raw_ECE": 0.011103651652698178,
      "calibrated_ECE": 0.010957860668963159,
      "selective_prediction": {
        "aurc": 0.0007221971521702212,
        "oracle_aurc": 7.51963918860967e-05,
        "excess_aurc": 0.0006470007602841245,
        "risk_at_80_coverage": 0.0013479957431713375,
        "risk_at_90_coverage": 0.0022072444917425405
      },
      "official_metrics": {
        "set": "xlsr_dev_calibrated",
        "n_files": 140950,
        "n_bonafide": 31334,
        "minDCF": 0.0222,
        "EER_%": 0.99,
        "CLLR": 0.0493,
        

### Fixed-coverage and per-slice selective analysis

Temperature scaling with positive T changes probability calibration but cannot change binary maximum-softmax confidence ordering: `max softmax(logits / T) = sigmoid(abs(logit_bonafide - logit_spoof) / T)`. Therefore ranking-based risk-coverage, AURC, and error at fixed coverage are invariant. The cell sorts each split once by the absolute logit difference to avoid artificial tie-order differences from floating-point division. It reports 80%, 90%, and 100% (no-abstention) coverage per attack and codec, plus overall bona-fide/spoof coverage and class-specific retained error. Attack AUC is the probability that a bona-fide example has a higher bona-fide-vs-spoof logit margin than an example from that attack; it is a separation statistic, not the within-attack selective error.

The analysis reads saved scores only; it does not rerun inference, feature caching, or training. XLS-R scores and calibration are separate from AASIST. For AASIST, this cell requires both `logit_spoof` and `logit_bonafide` and uses their difference for predictions, margins, and MSP confidence; it never substitutes the `cm-score` column. AASIST temperature is fit on dev only by binary NLL. If no finite positive temperature minimizes dev NLL, the fit report says so rather than silently using XLS-R's temperature.

A12 is in dev, so its slice results are descriptive rather than independent evidence; A18/A19 are eval slices. Attack slices contain spoof-only examples, so bona-fide coverage is undefined within those rows.

In [7]:
# Per-slice selective analysis from saved XLS-R and (optionally) AASIST score TSVs.
# No model inference, training, or feature caching is performed in this cell.
import json
import math
import os
import numpy as np
import pandas as pd
from scipy.optimize import brentq
from scipy.special import expit

with open(os.path.join(SSL_METRICS_DIR, 'xlsr_calibration_and_selective.json'), encoding='utf-8') as stream:
    _calibration = json.load(stream)
if _calibration.get('temperature_fitted_on') != 'dev':
    raise ValueError('Expected XLS-R temperature scaling to be fit on dev only.')
_slice_temperature = float(_calibration['temperature'])
if not np.isfinite(_slice_temperature) or _slice_temperature <= 0:
    raise ValueError(f'Temperature must be positive and finite; got {_slice_temperature}.')

# The supplied AASIST TSVs are in the project root; change this if they are moved.
AASIST_SCORE_DIR = PROJECT_DIR
_score_sources = {
    'XLS-R': (SSL_SCORE_DIR, 'xlsr'),
    'AASIST': (AASIST_SCORE_DIR, 'aasist'),
}


def _fit_binary_temperature(margins, labels):
    signed_margins = np.where(labels == 1, margins, -margins)

    def _nll(inverse_temperature):
        return float(np.logaddexp(0.0, -inverse_temperature * signed_margins).mean())

    def _gradient(inverse_temperature):
        return float(-np.mean(signed_margins * expit(-inverse_temperature * signed_margins)))

    if _gradient(0.0) >= 0.0:
        return {
            'temperature': None,
            'temperature_fitted_on': 'dev',
            'status': 'no_finite_optimum; dev NLL is minimized as temperature tends to infinity',
            'dev_nll_at_limit': _nll(0.0),
        }
    if not np.any(signed_margins < 0.0):
        return {
            'temperature': None,
            'temperature_fitted_on': 'dev',
            'status': 'no_finite_optimum; dev NLL decreases as temperature tends to zero',
            'dev_nll_at_limit': 0.0,
        }

    upper = 1.0
    while _gradient(upper) <= 0.0:
        upper *= 2.0
    inverse_temperature = brentq(_gradient, 0.0, upper)
    return {
        'temperature': float(1.0 / inverse_temperature),
        'temperature_fitted_on': 'dev',
        'status': 'finite_optimum',
        'dev_nll_at_optimum': _nll(inverse_temperature),
    }


def _auc_bonafide_vs_attack(bonafide_margins, attack_margins):
    n_bonafide = len(bonafide_margins)
    n_attack = len(attack_margins)
    combined_ranks = pd.Series(np.concatenate([bonafide_margins, attack_margins])).rank(
        method='average'
    ).to_numpy(dtype=np.float64)
    rank_sum_bonafide = combined_ranks[:n_bonafide].sum()
    return float(
        (rank_sum_bonafide - n_bonafide * (n_bonafide + 1) / 2)
        / (n_bonafide * n_attack)
    )


def _selective_slice_row(frame, model_name, split_name, slice_type, slice_name, coverage_pct):
    margins = frame['_margin'].to_numpy(dtype=np.float64)
    labels = frame['_label'].to_numpy(dtype=np.int64)
    errors = (margins >= 0.0) != labels
    order = np.argsort(-np.abs(margins), kind='stable')
    ordered_errors = errors[order].astype(np.float64)
    count = np.arange(1, len(frame) + 1)
    risk = np.cumsum(ordered_errors) / count
    coverage_curve = count / len(frame)
    retained_count = int(math.ceil(coverage_pct / 100.0 * len(frame)))
    retained = np.zeros(len(frame), dtype=bool)
    retained[order[:retained_count]] = True

    row = {
        'model': model_name,
        'split': split_name,
        'slice_type': slice_type,
        'slice': str(slice_name),
        'n_files': int(len(frame)),
        'n_retained': retained_count,
        'target_coverage_pct': coverage_pct,
        'actual_coverage_pct': 100.0 * retained_count / len(frame),
        'retained_error_pct': 100.0 * float(errors[retained].mean()),
        'aurc': float(np.trapezoid(np.r_[0.0, risk], np.r_[0.0, coverage_curve])),
    }
    for class_name, class_label in (('bonafide', 1), ('spoof', 0)):
        class_mask = labels == class_label
        class_retained = retained & class_mask
        class_total = int(class_mask.sum())
        row[f'n_{class_name}'] = class_total
        row[f'{class_name}_coverage_pct'] = (
            100.0 * int(class_retained.sum()) / class_total if class_total else np.nan
        )
        row[f'{class_name}_retained_error_pct'] = (
            100.0 * float(errors[class_retained].mean()) if class_retained.any() else np.nan
        )
    return row


_slice_rows = []
_invariance_rows = []
_missing_aasist = []
for _model_name, (_score_dir, _prefix) in _score_sources.items():
    for _split_name in ('dev', 'eval'):
        _score_path = os.path.join(_score_dir, f'{_prefix}_{_split_name}.tsv')
        if not os.path.isfile(_score_path):
            if _model_name == 'AASIST':
                _missing_aasist.append(_score_path)
                continue
            raise FileNotFoundError(f'Missing {_model_name} {_split_name} scores: {_score_path}')

        _scores = pd.read_csv(_score_path, sep='\t')
        _scores['filename'] = _scores['filename'].astype(str)
        _protocol = proto[_split_name][['filename', 'ATTACK_LABEL', 'CODEC', 'KEY']].copy()
        _protocol['filename'] = _protocol['filename'].astype(str)
        if _protocol['filename'].duplicated().any() or _scores['filename'].duplicated().any():
            raise ValueError(f'{_model_name} {_split_name} protocol and score filenames must be unique.')

        if not {'logit_spoof', 'logit_bonafide'}.issubset(_scores.columns):
            raise ValueError(
                f'{_model_name} {_split_name} scores must include logit_spoof and '
                'logit_bonafide; cm-score alone is not the binary logit difference needed for MSP.'
            )
        _scores['_margin'] = (
            _scores['logit_bonafide'].to_numpy(dtype=np.float64)
            - _scores['logit_spoof'].to_numpy(dtype=np.float64)
        )

        _joined = _protocol.merge(
            _scores[['filename', '_margin']], on='filename', how='outer',
            validate='one_to_one', indicator=True,
        )
        if not (_joined['_merge'] == 'both').all():
            _extra_scores = int((_joined['_merge'] == 'right_only').sum())
            _missing_scores = int((_joined['_merge'] == 'left_only').sum())
            raise ValueError(
                f'{_model_name} {_split_name} score/protocol mismatch: '
                f'{_extra_scores} extra scores, {_missing_scores} missing scores.'
            )
        _joined = _joined.drop(columns='_merge')
        if not set(_joined['KEY'].unique()).issubset({'bonafide', 'spoof'}):
            raise ValueError(f'Unexpected {_split_name} protocol labels.')
        _joined['_label'] = (_joined['KEY'] == 'bonafide').astype(np.int64)
        _margin = _joined['_margin'].to_numpy(dtype=np.float64)
        if not np.isfinite(_margin).all():
            raise ValueError(f'{_model_name} {_split_name} score margins contain non-finite values.')

        if _model_name == 'AASIST' and _split_name == 'dev':
            _aasist_temperature_fit = _fit_binary_temperature(
                _margin, _joined['_label'].to_numpy(dtype=np.int64)
            )
        _model_temperature = _slice_temperature if _model_name == 'XLS-R' else _aasist_temperature_fit.get('temperature')
        _order = np.argsort(-np.abs(_margin), kind='stable')
        _labels = _joined['_label'].to_numpy(dtype=np.int64)
        _ordered_errors = (((_margin >= 0.0) != _labels)[_order]).astype(np.float64)
        _count = np.arange(1, len(_labels) + 1)
        _coverage = _count / len(_labels)
        _risk = np.cumsum(_ordered_errors) / _count
        _aurc = float(np.trapezoid(np.r_[0.0, _risk], np.r_[0.0, _coverage]))
        _invariance_rows.append({
            'model': _model_name,
            'split': _split_name,
            'temperature': _model_temperature,
            'temperature_fitted_on': 'dev',
            'confidence_order_invariant_by_positive_temperature': True,
            'raw_aurc': _aurc,
            'temperature_scaled_aurc': _aurc,
            'fixed_coverage_error_invariant_by_positive_temperature': True,
            'temperature_fit_status': (
                'existing_xlsr_dev_fit' if _model_name == 'XLS-R'
                else _aasist_temperature_fit['status']
            ),
        })

        for _pct in (80, 90, 100):
            _slice_rows.append(
                _selective_slice_row(_joined, _model_name, _split_name, 'overall', 'all', _pct)
            )
        for _column, _slice_type in (('ATTACK_LABEL', 'attack'), ('CODEC', 'codec')):
            for _slice_name, _group in _joined.groupby(_column, sort=True, dropna=False):
                _attack_auc = np.nan
                if _slice_type == 'attack':
                    _attack_spoof_margins = _group.loc[_group['_label'] == 0, '_margin'].to_numpy(dtype=np.float64)
                    if len(_attack_spoof_margins) and (_group['_label'] == 1).sum() == 0:
                        _bonafide_margins = _joined.loc[_joined['_label'] == 1, '_margin'].to_numpy(dtype=np.float64)
                        _attack_auc = _auc_bonafide_vs_attack(_bonafide_margins, _attack_spoof_margins)
                for _pct in (80, 90, 100):
                    _row = _selective_slice_row(
                        _group, _model_name, _split_name, _slice_type, _slice_name, _pct
                    )
                    _row['auc_bonafide_vs_attack'] = _attack_auc
                    _slice_rows.append(_row)

_slice_report = pd.DataFrame(_slice_rows)
_slice_path = os.path.join(SSL_METRICS_DIR, 'per_model_slice_selective.tsv')
_slice_report.to_csv(_slice_path, sep='\t', index=False)
_invariance_path = os.path.join(SSL_METRICS_DIR, 'selective_ranking_invariance.json')
with open(_invariance_path, 'w', encoding='utf-8') as stream:
    json.dump(_invariance_rows, stream, indent=2)
_aasist_temperature_path = os.path.join(SSL_METRICS_DIR, 'aasist_temperature_fit.json')
with open(_aasist_temperature_path, 'w', encoding='utf-8') as stream:
    json.dump(_aasist_temperature_fit, stream, indent=2)

print('Temperature scaling changes probability calibration, not confidence ranking; positive-temperature invariance is mathematical.')
print('AASIST temperature fit on dev:', _aasist_temperature_fit)
print(pd.DataFrame(_invariance_rows).to_string(index=False))
print('\nTarget attack slices: A12 is dev; A18/A19 are eval. A12 is descriptive, not independent evidence.')
display(_slice_report[
    (_slice_report['slice_type'] == 'attack')
    & (_slice_report['slice'].isin(['A12', 'A18', 'A19']))
].reset_index(drop=True))
if _missing_aasist:
    print('AASIST slices were skipped because these score files are not available locally:')
    for _missing_path in _missing_aasist:
        print(' -', _missing_path)
    print('Set AASIST_SCORE_DIR to the local folder containing aasist_dev.tsv and aasist_eval.tsv, then rerun this cell.')
print(f'Full overall, attack, and codec tables saved to: {_slice_path}')
print(f'Ranking-invariance check saved to: {_invariance_path}')
print(f'AASIST temperature-fit report saved to: {_aasist_temperature_path}')

Temperature scaling changes probability calibration, not confidence ranking; positive-temperature invariance is mathematical.
AASIST temperature fit on dev: {'temperature': None, 'temperature_fitted_on': 'dev', 'status': 'no_finite_optimum; dev NLL is minimized as temperature tends to infinity', 'dev_nll_at_limit': 0.6931471805599452}
 model split  temperature temperature_fitted_on  confidence_order_invariant_by_positive_temperature  raw_aurc  temperature_scaled_aurc  fixed_coverage_error_invariant_by_positive_temperature                                                   temperature_fit_status
 XLS-R   dev     1.522302                   dev                                                True  0.000722                 0.000722                                                    True                                                    existing_xlsr_dev_fit
 XLS-R  eval     1.522302                   dev                                                True  0.008971                 0.008971 

,model,split,slice_type,slice,n_files,n_retained,target_coverage_pct,actual_coverage_pct,retained_error_pct,aurc,n_bonafide,bonafide_coverage_pct,bonafide_retained_error_pct,n_spoof,spoof_coverage_pct,spoof_retained_error_pct,auc_bonafide_vs_attack
0,XLS-R,dev,attack,A12,13702,10962,80,80.002919,0.009122,0.000375,0,NaN,NaN,13702,80.002919,0.009122,0.998840
1,XLS-R,dev,attack,A12,13702,12332,90,90.001460,0.089199,0.000375,0,NaN,NaN,13702,90.001460,0.089199,0.998840
2,XLS-R,dev,attack,A12,13702,13702,100,100.000000,1.145818,0.000375,0,NaN,NaN,13702,100.000000,1.145818,0.998840
3,XLS-R,eval,attack,A18,34368,27495,80,80.001746,1.356610,0.008475,0,NaN,NaN,34368,80.001746,1.356610,0.953110
4,XLS-R,eval,attack,A18,34368,30932,90,90.002328,3.158541,0.008475,0,NaN,NaN,34368,90.002328,3.158541,0.953110
5,XLS-R,eval,attack,A18,34368,34368,100,100.000000,6.677723,0.008475,0,NaN,NaN,34368,100.000000,6.677723,0.953110
6,XLS-R,eval,attack,A19,30860,24688,80,80.000000,0.000000,0.000037,0,NaN,NaN,30860,80.000000,0.000000,0.993554
7,XLS-R,eval,attack,A19,30860,27774,90,90.000000,0.000000,0.000037,0,NaN,NaN,30860,90.000000,0.000000,0.993554
8,XLS-R,eval,attack,A19,30860,30860,100,100.000000,0.430979,0.000037,0,NaN,NaN,30860,100.000000,0.430979,0.993554
9,AASIST,dev,attack,A12,13702,10962,80,80.002919,100.000000,0.999964,0,NaN,NaN,13702,80.002919,100.000000,0.193745


Full overall, attack, and codec tables saved to: c:\Users\ghazz\Downloads\DeepLearningProj\outputs\shriya_aasist\ssl_detector\metrics\per_model_slice_selective.tsv
Ranking-invariance check saved to: c:\Users\ghazz\Downloads\DeepLearningProj\outputs\shriya_aasist\ssl_detector\metrics\selective_ranking_invariance.json
AASIST temperature-fit report saved to: c:\Users\ghazz\Downloads\DeepLearningProj\outputs\shriya_aasist\ssl_detector\metrics\aasist_temperature_fit.json
